In [1]:
# Cell 1: Imports
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Cell 2: Create synthetic training dataset
# In production, this comes from anonymized IEPs

data= {
    'grade_level':[3,4,3,5,2,4,3,5,4,3] * 10, # Grades 2-5
    'primary_diagnosis':['ADHD', 'Dyslexia', 'ADHD', 'Dyscalculia', 'Autism'
                        , 'ADHD', 'Dyslexia', 'Autism', 'ADHD', 'Dyslexia']
                        *10,
    'reading_fluency':[2,1,3,3,2,2,1,3,2,1]*10, # 1=low, 2=average, 3=above avg
    'math_skill':[2, 3, 1, 1, 2, 3, 3, 2, 2, 3] * 10,
    'attention_level': [1, 3, 1, 3, 2, 1, 3, 2, 1, 3] * 10,  # 1=low, 2=average, 3=high
    'social_skills': [2, 2, 2, 2, 1, 3, 2, 1, 2, 2] * 10,
    'motor_skills': [3, 2, 3, 2, 2, 3, 2, 2, 3, 2] * 10,
    'accommodations': [
        'Extended time,Frequent breaks,Quiet space',  # ADHD with low reading
        'Extended time,Large print,Text-to-speech',   # Dyslexia
        'Extended time,Frequent breaks',              # ADHD avg reading
        'Calculator use,Extended time',               # Dyscalculia
        'Quiet space,Movement breaks,Fidget tools',   # Autism
        'Extended time,Frequent breaks',              # ADHD
        'Extended time,Large print',                  # Dyslexia with low fluency
        'Movement breaks,Noise-cancelling,Transition time',  # Autism
        'Extended time,Breaks',                       # ADHD
        'Extended time,Text-to-speech'                # Dyslexia
    ] * 10
}

In [3]:
# data frame is like a spreadsheet. it's just columns and rows.  
# The 'Shape' is its representation on a screen.

df = pd.DataFrame(data)
print(f"Dataset shape: {df.shape}")
print(df.head())

Dataset shape: (100, 8)
   grade_level primary_diagnosis  reading_fluency  math_skill  \
0            3              ADHD                2           2   
1            4          Dyslexia                1           3   
2            3              ADHD                3           1   
3            5       Dyscalculia                3           1   
4            2            Autism                2           2   

   attention_level  social_skills  motor_skills  \
0                1              2             3   
1                3              2             2   
2                1              2             3   
3                3              2             2   
4                2              1             2   

                              accommodations  
0  Extended time,Frequent breaks,Quiet space  
1   Extended time,Large print,Text-to-speech  
2              Extended time,Frequent breaks  
3               Calculator use,Extended time  
4   Quiet space,Movement breaks,Fidget tool

In [4]:
# Cell 3: Split accommodations into multilabel format
from sklearn.preprocessing import MultiLabelBinarizer

# Parse comma-separated accommodations
df['accommodations_list'] = df['accommodations'].apply(
    lambda x: [a.strip() for a in x.split(',')]
)

# Get all unique accommodations.  A set can have only one of each, it will
# naturally ignore duplicates.
all_accommodations = set()
for accs in df['accommodations_list']:
    all_accommodations.update(accs)
all_accommodations = sorted(list(all_accommodations))
print(f"\nAll accommodations: ")
for acc in all_accommodations:
    print(f" {acc}")


All accommodations: 
 Breaks
 Calculator use
 Extended time
 Fidget tools
 Frequent breaks
 Large print
 Movement breaks
 Noise-cancelling
 Quiet space
 Text-to-speech
 Transition time


In [6]:
# Create multi-label matrix
# we need a multi-label matrix because it's how the machine-learning model
# digests the accommodations.  "fit_transform" is it's way of translating
# the list of accommodations into ones and zeros.

mlb= MultiLabelBinarizer()
y=mlb.fit_transform(df['accommodations_list'])

# the multi-label "matrix" is a NumPy array.
print(f"Multilabel matrix shape: {y.shape}")


# Convert binary matrix to DataFrame for better readability
y_df = pd.DataFrame(y, columns=mlb.classes_)
print("multilabel matrix (first 10 rows):")
print (y_df.head(10))




Multilabel matrix shape: (100, 11)
multilabel matrix (first 10 rows):
   Breaks  Calculator use  Extended time  Fidget tools  Frequent breaks  \
0       0               0              1             0                1   
1       0               0              1             0                0   
2       0               0              1             0                1   
3       0               1              1             0                0   
4       0               0              0             1                0   
5       0               0              1             0                1   
6       0               0              1             0                0   
7       0               0              0             0                0   
8       1               0              1             0                0   
9       0               0              1             0                0   

   Large print  Movement breaks  Noise-cancelling  Quiet space  \
0            0                0       

In [7]:
# Cell 4: Prepare features
# These are the input features, student characteristics we give the ML model
# to see if it can guess the accommodations.  
X = df[['grade_level', 'reading_fluency', 'math_skill', 'attention_level'
       , 'social_skills', 'motor_skills']].copy()

In [8]:
# Encode diagnosis
# The model uses numbers, but diagnoses are letters/words
le = LabelEncoder()
diagnosis_encoded = le.fit_transform(df['primary_diagnosis'])
X['diagnosis_encoded']=diagnosis_encoded

print(f"\nFeature matrix:\n{X.head()}")
print(f"Feature shape: {X.shape}")


Feature matrix:
   grade_level  reading_fluency  math_skill  attention_level  social_skills  \
0            3                2           2                1              2   
1            4                1           3                3              2   
2            3                3           1                1              2   
3            5                3           1                3              2   
4            2                2           2                2              1   

   motor_skills  diagnosis_encoded  
0             3                  0  
1             2                  3  
2             3                  0  
3             2                  2  
4             2                  1  
Feature shape: (100, 7)


In [9]:
# Cell 5: Train-test split
Features_train, Features_test, labels_train, labels_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {Features_train.shape[0]}, Test size: {Features_test.shape[0]}")

Train size: 80, Test size: 20


In [10]:
# Cell 6: Train models (one per accommodation - multilabel approach)
models = {}
for idx, accommodation in enumerate(all_accommodations):
    model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    model.fit(Features_train, labels_train[:, idx])
    accuracy = model.score(Features_test, labels_test[:, idx])
    models[accommodation]=model
    print(f"{accommodation}: {accuracy:.2%} accuracy")
    

Breaks: 100.00% accuracy
Calculator use: 100.00% accuracy
Extended time: 100.00% accuracy
Fidget tools: 100.00% accuracy
Frequent breaks: 100.00% accuracy
Large print: 95.00% accuracy
Movement breaks: 100.00% accuracy
Noise-cancelling: 100.00% accuracy
Quiet space: 100.00% accuracy
Text-to-speech: 95.00% accuracy
Transition time: 100.00% accuracy


In [11]:
# Cell 7: Predict for a new student
def predict_accommodations(grade, diagnosis, reading, math, attention,
                           social, motor, le):
    # Create feature vector
    diagnosis_code=le.transform([diagnosis])[0]
    student_features=pd.DataFrame({
        'grade_level':[grade],
        'reading_fluency':[reading],
        'math_skill':[math],
        'attention_level':[attention],
        'social_skills':[social],
        'motor_skills':[motor],
        'diagnosis_encoded':[diagnosis_code]
    })
    # Predict
    predictions={}
    for accommodation, model in models.items():
        probability= model.predict_proba(student_features)[0][1]
        predictions[accommodation] = probability
        
    # Sort by probability and return top 4
    sorted_preds=sorted(predictions.items(), key=lambda x:x[1], reverse=True)
    return [(acc, prob) for acc, prob in sorted_preds[:4]]

In [12]:
# Test prediction
print("\n---SAMPLE PREDICTION----")
result = predict_accommodations(
        grade=3,
        diagnosis='ADHD',
        reading=2, #average
        math=2,
        attention=1, #low
        social=2,
        motor=3,
        le=le
    )
print("Predicted accommodations for Grade 3 ADHD student:")
for acc, prob in result:
    print(f" {acc}: {prob:1%} confidence")


---SAMPLE PREDICTION----
Predicted accommodations for Grade 3 ADHD student:
 Extended time: 100.000000% confidence
 Frequent breaks: 100.000000% confidence
 Quiet space: 99.000000% confidence
 Breaks: 0.000000% confidence


In [13]:
# Cell 8: Evaluate model performance
Features_test_df= pd.DataFrame(Features_test, columns=Features_test.columns)

y_pred_all = np.array([
    [models[acc].predict(Features_test_df)[i] for acc in all_accommodations]
    for i in range(len(Features_test_df))
])

In [16]:
# Calculate hamming loss (multilabel metric)
from sklearn.metrics import hamming_loss, jaccard_score
print(f"\nModel Performance:")
print(f"Hamming Loss: {hamming_loss(labels_test, y_pred_all):.3f}")
print(f"Jaccard Score: {jaccard_score(labels_test, y_pred_all, average='micro'):.3f}")


Model Performance:
Hamming Loss: 0.009
Jaccard Score: 0.961


In [17]:
# Cell 9: Export to pickle for later use
import pickle
with open('accommodation_model.pkl', 'wb') as f:
    pickle.dump({
        'models':models,
        'mlb':mlb,
        'label_encoder':le,
        'accommodations': all_accommodations
    }, f)
    print("Model saved to accommodation_model.pkl")
    
    

Model saved to accommodation_model.pkl


In [19]:
# Cell 10: Export to Core ML format (for Swift app)
import coremltools as ct

# For Core ML, we'll export a simplified single model
# Create a combined model that outputs accommodation predictions
X_train_np = Features_train.values

labels_train_combined = labels_train

# Train a single model for demonstration
main_model = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42)
main_model.fit(X_train_np, labels_train_combined)

# Convert the first model as a demonstration
first_accommodation = all_accommodations[0]
demo_model = models[first_accommodation]

try:
    ml_model = ct.converters.sklearn.convert(
        demo_model,
        input_features=['grade_level', 'reading_fluency', 'math_skill', 'attention_level', 'social_skills', 'motor_skills', 'diagnosis_encoded']
    )
    ml_model.save(f'AccommodationDemo_{first_accommodation.replace(" ", "_")}.mlmodel')
    print(f"Demo Core ML model saved for {first_accommodation}")
except Exception as e:
    print(f"Core ML conversion note: {e}")
    print("Use Python backend instead for full multilabel support")

Core ML conversion note: name '_tree' is not defined
Use Python backend instead for full multilabel support
